In [3]:
import warnings
import pandas as pd
import numpy as np
from itertools import product
from datetime import date

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

transaction_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Grocery_Refund\Contacts\Grocery_refund_contact.csv"
df = pd.read_csv(transaction_path, low_memory=False)

df[df['marketplace_id']!="marketplace_id"]
df['session_dt'] = df['session_dt'].astype(str).str.replace(r'[\x00-\x1F\u200B-\u200F\uFEFF\u00A0]', '', regex=True).str.strip()


df["session_dt"] = pd.to_datetime(df["session_dt"], format="%d-%m-%Y", errors="coerce")
df["session_dt_parsed"] = df["session_dt"].dt.date 

Jn_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\JN_mapping.xlsx"
df2= pd.read_excel(Jn_path)
merged_df  = pd.merge(df,df2, how ="left", left_on ="ssi", right_on ="current_iss")

SSI_Path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Refund_Mapping_ssi.xlsx"
df4= pd.read_excel(SSI_Path)
merged_df  = pd.merge(merged_df,df4, how ="left", left_on ="ssi", right_on ="current_iss")

df["session_dt"] = pd.to_datetime(df["session_dt"].astype(str).str.strip(), errors="coerce")
df["session_dt_parsed"] = df["session_dt"].dt.date  

Conversation_factor= r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Conversation_Factors.xlsx"
df3 =pd.read_excel(Conversation_factor)

merged_df["contacts"] =pd.to_numeric(merged_df["contacts"], errors='coerce')
merged_df["session_dt"] = pd.to_datetime(merged_df["session_dt"], errors='coerce', format='%d-%m-%Y')
##merged_df["session_dt_parsed"] = merged_df["session_dt"].dt.date 
merged_df["session_dt_parsed"] = merged_df["session_dt"].apply(lambda x: x.date() if pd.notna(x) else 0)





filtered_data= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) ].copy()

total_contacts = (filtered_data.groupby("session_dt", observed=False)["contacts"].sum().reset_index()
                    .rename(columns={"contacts": "total_contacts"}))


refund_ssi = ["Flipkart Pay Later->Payment Issues->Flipkart Pay Later - Refund not adjusted to bill",
"Order Delivery->Order status related->I have Service center/warranty related issues",
"Refund Requests (CS Raised)->Token of Apology->NA",
"Refunds->Refund Related Enquiry->Refund Status Enquiry_HL",
"Refunds->Refund Related Issues->Cx wants to callback later",
"Refunds->Refund Related Issues->Eligible TOA not provided",
"Refunds->Refund Related Issues->I have refund related queries",
"Refunds->Refund Related Issues->Installation Complete - No damage reported",
"Refunds->Refund Related Issues->Installation Pending - Request Cx to Wait",
"Refunds->Refund Related Issues->Partial Refund Initiated_HL",
"Travel->Refund Related->Travel - Refund Related",
"Refund Follow-ups->CB - NEFT Details->NA",
"Refunds->Refund Related Issues->Refund credit failed"
]
merged_df["ssi"]=merged_df["ssi"].apply(
    lambda x:"refund_ssi" if x in refund_ssi else x
)

transaction_order = ["Refunds->Refund Related Enquiry->Refund Status Enquiry",
"Refunds->Refund Related Issues->Refund stuck_Int/processing",
"Refunds->Refund Related Issues->Refund completed but not reflecting",
"Refunds->Refund Related Issues->Refund Failed/cancelled",
"Refund Follow-ups->Gathered Offline Refund Details->NA",
"Refunds->Refund Related Issues->Cancellation done but refund pending",
"Flipkart Plus->Plus Price->Issues with Plus Price coins refund",
"Refunds->Refund Related Issues->Refund Status Related",
"Refunds->Refund Related Issues->Return Override",
 "refund_ssi"                    
 ]
filtered_data1= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL","RCBP","FLIPKART"])) &
                                        (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                                         (merged_df["JN"]=="REFUNDS")].copy()
filtered_data1["ssi"]= pd.Categorical(filtered_data1["ssi"], categories =transaction_order, ordered=True )

refund_contacts_summary = (
    filtered_data1.groupby(["ssi","session_dt"], observed =False)["contacts"].sum().reset_index()
    .rename(columns={"contacts":"refund_contacts_count" })
)

pivot_refund_countact = refund_contacts_summary.pivot_table(
    index="ssi",
    columns="session_dt",  # This is datetime
    values="refund_contacts_count",
    aggfunc="sum",
    observed=False
).fillna(0)



pivot_refund_countact.columns=pd.to_datetime(pivot_refund_countact.columns, errors='coerce')
pivot_refund_countact = pivot_refund_countact.sort_index(axis=1)
pivot_refund_countact.columns = pivot_refund_countact.columns.strftime('%d-%m-%Y')
pivot_refund_countact.reset_index(inplace=True)

df3["session_dt"]=pd.to_datetime(df3["Date"], format="%d-%m-%Y", errors= "coerce")
conversion_df = pd.merge(df3,total_contacts, on="session_dt", how ="inner")
conversion_df["conversion_pct"]= (conversion_df["Convertion_factor"]/conversion_df["total_contacts"]).round(2)
conversion_df["date_str"]= conversion_df["session_dt"].dt.strftime('%d-%m-%Y')
conversion_map = dict(zip(conversion_df["date_str"],conversion_df["conversion_pct"]))

for col in pivot_refund_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_countact[col] = pivot_refund_countact[col] * conversion_map[col]

# Final rounding
pivot_refund_countact.iloc[:, 1:] = pivot_refund_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_transaction_source_countact.head(10))



##-----------------------------------------------#########
##Refunnd_reason_flag_contacts

merged_df["refund_reason_flag"]=merged_df["refund_reason_flag"].apply(
    lambda x:"RTO_Cancellation" if x in ["RTO","Cancellation"] 
    else x if x in ["Return","Adonc"]
    else "Others")

filtered_data1= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                                        (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                                        (merged_df["SSI_Flag"]=="YES") &
                                      (merged_df["JN"]=="REFUNDS")].copy()   

transaction_order1 = ["RTO_Cancellation","Return", "Adonc","Others"]
filtered_data1["refund_reason_flag"] =pd.Categorical(filtered_data1["refund_reason_flag"], categories=transaction_order1, ordered=True)

refund_reason_flag_summary = (filtered_data1.groupby(["refund_reason_flag","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_reason_count"}))

pivot_refund_reason_countact = refund_reason_flag_summary.pivot_table(index="refund_reason_flag",
                                                                     columns="session_dt",
                                                                     values = "refund_reason_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_refund_reason_countact.columns=pd.to_datetime(pivot_refund_reason_countact.columns, errors='coerce')
pivot_refund_reason_countact = pivot_refund_reason_countact.sort_index(axis=1)
pivot_refund_reason_countact.columns = pivot_refund_reason_countact.columns.strftime('%d-%m-%Y')
pivot_refund_reason_countact.reset_index(inplace=True)

for col in pivot_refund_reason_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_reason_countact[col] = pivot_refund_reason_countact[col] * conversion_map[col]

# Final rounding
pivot_refund_reason_countact.iloc[:, 1:] = pivot_refund_reason_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_refund_reason_countact.head(10))


###-----------------------------------------------------------------------------#####
##Payment_type_final##
merged_df["payment_type_final"]=merged_df["payment_type_final"].apply(
    lambda x: x if x in ['Pre-paid','COD','Multi-payment', 'EGV/Wallet/Coin']
     else "Others")

filtered_data2= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                            (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                            (merged_df["SSI_Flag"]=="YES") &
                          (merged_df["JN"]=="REFUNDS")].copy()   

transaction_order2 = ['Pre-paid','COD','Multi-payment', 'Others', 'EGV/Wallet/Coin']
filtered_data2["payment_type_final"] =pd.Categorical(filtered_data2["payment_type_final"], categories=transaction_order2, ordered=True)

refund_payment_type_summary = (filtered_data2.groupby(["payment_type_final","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_payment_type_final_count"}))
pivot_refund_payment_type_countact = refund_payment_type_summary.pivot_table(index="payment_type_final",
                                                                     columns="session_dt",
                                                                     values = "refund_payment_type_final_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_refund_payment_type_countact.columns=pd.to_datetime(pivot_refund_payment_type_countact.columns, errors='coerce')
pivot_refund_payment_type_countact = pivot_refund_payment_type_countact.sort_index(axis=1)
pivot_refund_payment_type_countact.columns = pivot_refund_payment_type_countact.columns.strftime('%d-%m-%Y')
pivot_refund_payment_type_countact.reset_index(inplace=True)

for col in pivot_refund_payment_type_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_payment_type_countact[col] = pivot_refund_payment_type_countact[col] * conversion_map[col]

# Final rounding
pivot_refund_payment_type_countact.iloc[:, 1:] = pivot_refund_payment_type_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_transaction_source_countact.head(10))

###-----------------------------------------------------------------------------#####
##ARN_ISSUE_FLAG##

filtered_data3= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                           (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                            (merged_df["SSI_Flag"]=="YES") &
                          (merged_df["JN"]=="REFUNDS")].copy()   

transaction_order3 = ["ARN_GENERATED","ARN_NOT_GENERATED", "ARN_NOT_APPLICABLE"]
filtered_data3["ARN_ISSUE_FLAG"] =pd.Categorical(filtered_data3["ARN_ISSUE_FLAG"], categories=transaction_order3, ordered=True)

refund_ARN_ISSUE_summary = (filtered_data3.groupby(["ARN_ISSUE_FLAG","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_ARN_ISSUE_count"}))
pivot_refund_ARN_ISSUE_countact = refund_ARN_ISSUE_summary.pivot_table(index="ARN_ISSUE_FLAG",
                                                                     columns="session_dt",
                                                                     values = "refund_ARN_ISSUE_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_refund_ARN_ISSUE_countact.columns=pd.to_datetime(pivot_refund_ARN_ISSUE_countact.columns, errors='coerce')
pivot_refund_ARN_ISSUE_countact = pivot_refund_ARN_ISSUE_countact.sort_index(axis=1)
pivot_refund_ARN_ISSUE_countact.columns = pivot_refund_ARN_ISSUE_countact.columns.strftime('%d-%m-%Y')
pivot_refund_ARN_ISSUE_countact.reset_index(inplace=True)

for col in pivot_refund_ARN_ISSUE_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_ARN_ISSUE_countact[col] = pivot_refund_ARN_ISSUE_countact[col] * conversion_map[col]

# Final rounding
pivot_refund_ARN_ISSUE_countact.iloc[:, 1:] = pivot_refund_ARN_ISSUE_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_refund_ARN_ISSUE_countact.head(10))

###-----------------------------------------------------------------------------#####
##ARN_ISSUE_FLAG_SSI##

arn_issues = [
    "Refunds->Refund Related Enquiry->Refund Status Enquiry",
    "Refunds->Refund Related Issues->Refund Failed/cancelled",
    "Refund Follow-ups->Gathered Offline Refund Details->NA",
    "Refunds->Refund Related Issues->Refund completed but not reflecting"
]
merged_df1 = merged_df.copy()
merged_df1["ssi"] = merged_df1["ssi"].apply(
lambda x: x if x in arn_issues else "Others"
)

filtered_data4= merged_df1[(~merged_df1["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                            (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                            (merged_df["SSI_Flag"]=="YES") &
                           (merged_df1["JN"]=="REFUNDS") &
                          (merged_df1["ssi"].isin(arn_issues))].copy()   

transaction_order4 = ["Refunds->Refund Related Enquiry->Refund Status Enquiry", "Refunds->Refund Related Issues->Refund Failed/cancelled",
    "Refund Follow-ups->Gathered Offline Refund Details->NA", "Refunds->Refund Related Issues->Refund completed but not reflecting"]

filtered_data4["ssi"] =pd.Categorical(filtered_data4["ssi"], categories=transaction_order4, ordered=True)


refund_ARN_ISSUE_SSI_summary = (filtered_data4.groupby(["ssi","ARN_ISSUE_FLAG","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_ARN_ISSUE_SSI_count"}))
ARN_NOT_GENERATED_df = refund_ARN_ISSUE_SSI_summary[refund_ARN_ISSUE_SSI_summary["ARN_ISSUE_FLAG"]=="ARN_NOT_GENERATED"]


pivot_refund_ARN_ISSUE_SSI_countact = ARN_NOT_GENERATED_df.pivot_table(index=["ssi"],
                                                                     columns="session_dt",
                                                                     values = "refund_ARN_ISSUE_SSI_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_refund_ARN_ISSUE_SSI_countact.columns=pd.to_datetime(pivot_refund_ARN_ISSUE_SSI_countact.columns, errors='coerce')
pivot_refund_ARN_ISSUE_SSI_countact = pivot_refund_ARN_ISSUE_SSI_countact.sort_index(axis=1)
pivot_refund_ARN_ISSUE_SSI_countact.columns = pivot_refund_ARN_ISSUE_SSI_countact.columns.strftime('%d-%m-%Y')
pivot_refund_ARN_ISSUE_SSI_countact.reset_index(inplace=True)

for col in pivot_refund_ARN_ISSUE_SSI_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_ARN_ISSUE_SSI_countact[col] = pivot_refund_ARN_ISSUE_SSI_countact[col] * conversion_map[col]

# Final rounding
pivot_refund_ARN_ISSUE_SSI_countact.iloc[:, 1:] = pivot_refund_ARN_ISSUE_SSI_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_refund_ARN_ISSUE_SSI_countact.head(10))


###-----------------------------------------------------------------------------#####
##refund_void##

filtered_data5= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                           (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                            (merged_df["SSI_Flag"]=="YES") &
                          (merged_df["JN"]=="REFUNDS")].copy()   

transaction_order5 = ["void_transaction","non_void"]
filtered_data5["void_flag"] =pd.Categorical(filtered_data5["void_flag"], categories=transaction_order5, ordered=True)

refund_Void_summary = (filtered_data5.groupby(["void_flag","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_Void_count"}))

pivot_refund_Void_countact = refund_Void_summary.pivot_table(index="void_flag",
                                                                     columns="session_dt",
                                                                     values = "refund_Void_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_refund_Void_countact.columns=pd.to_datetime(pivot_refund_Void_countact.columns, errors='coerce')
pivot_refund_Void_countact = pivot_refund_Void_countact.sort_index(axis=1)
pivot_refund_Void_countact.columns = pivot_refund_Void_countact.columns.strftime('%d-%m-%Y')
pivot_refund_Void_countact.reset_index(inplace=True)

for col in pivot_refund_Void_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_Void_countact[col] = pivot_refund_Void_countact[col] * conversion_map[col]

# Final rounding
pivot_refund_Void_countact.iloc[:, 1:] = pivot_refund_Void_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_refund_Void_countact.head(10))


###-----------------------------------------------------------------------------#####
##refund_mode_SSI##

merged_df2 = merged_df.copy()

merged_df2["refund_mode"] = merged_df2["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data6= merged_df2[(~merged_df2["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) & 
                            (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                            (merged_df["SSI_Flag"]=="YES") &
                            (merged_df2["JN"]=="REFUNDS")].copy()   
    
transaction_order6 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","FLIPKART_FINANCE","InvalidRefundMode","LOAN","SUPERPAY_LATER","Others"]

filtered_data6["refund_mode"] =pd.Categorical(filtered_data6["refund_mode"], categories=transaction_order6, ordered=True)

refund_mode_ssi_summary = (filtered_data6.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi_count"}))
refund_mode_ssi_summary["session_dt"] = pd.to_datetime(refund_mode_ssi_summary["session_dt"], errors='coerce')
all_dates = pd.to_datetime(filtered_data6["session_dt"].dropna().unique())

all_combinations = pd.DataFrame(list(product(transaction_order6, all_dates)), columns=["refund_mode","session_dt" ])
all_combinations["session_dt"] = pd.to_datetime(all_combinations["session_dt"], errors='coerce')


complete_summary = all_combinations.merge(
    refund_mode_ssi_summary,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi_countact = complete_summary.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_refund_mode_ssi_countact.columns=pd.to_datetime(pivot_refund_mode_ssi_countact.columns, errors='coerce')
pivot_refund_mode_ssi_countact = pivot_refund_mode_ssi_countact.sort_index(axis=1)
pivot_refund_mode_ssi_countact.columns = pivot_refund_mode_ssi_countact.columns.strftime('%d-%m-%Y')
pivot_refund_mode_ssi_countact.reset_index(inplace=True)

pivot_refund_mode_ssi_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi_countact["refund_mode"],
	categories = transaction_order6,
	ordered =True)

pivot_refund_mode_ssi_countact = pivot_refund_mode_ssi_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi_countact[col] = pivot_refund_mode_ssi_countact[col] * conversion_map[col]

# Final rounding
pivot_refund_mode_ssi_countact.iloc[:, 1:] = pivot_refund_mode_ssi_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_refund_mode_ssi_countact.head(10))

output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Grocery_Refund_contacts.xlsx"
with pd.ExcelWriter(output_path, engine = 'openpyxl') as writer:
    pivot_refund_countact.to_excel(writer, sheet_name ='refund_countact', index=False)
    pivot_refund_reason_countact.to_excel(writer, sheet_name ='refund_reason_countact', index=False )
    pivot_refund_payment_type_countact.to_excel(writer, sheet_name ='refund_payment_type', index=False )
    pivot_refund_ARN_ISSUE_countact.to_excel(writer, sheet_name = 'ARN_ISSUE_countact', index=False)
    pivot_refund_ARN_ISSUE_SSI_countact.to_excel(writer, sheet_name = 'ARN_ISSUE_SSI', index=False)
    pivot_refund_Void_countact.to_excel(writer, sheet_name = 'refund_Void_countact', index=False)
    pivot_refund_mode_ssi_countact.to_excel(writer, sheet_name = 'mode_ssi_countact', index=False)
print("Excel file successfully saved:",output_path)

Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Grocery_Refund_contacts.xlsx


In [5]:
print(conversion_df)

         Date  Convertion_factor session_dt  total_contacts  conversion_pct  \
0  2025-04-01             199324 2025-04-01          141395            1.41   
1  2025-04-02             205988 2025-04-02          147710            1.39   
2  2025-04-03             213555 2025-04-03          149809            1.43   
3  2025-04-04             213603 2025-04-04          153993            1.39   
4  2025-04-05             209985 2025-04-05          151682            1.38   
..        ...                ...        ...             ...             ...   
64 2025-06-04             214421 2025-06-04          164689            1.30   
65 2025-06-05             220869 2025-06-05          171815            1.29   
66 2025-06-06             213425 2025-06-06          167170            1.28   
67 2025-06-07             196198 2025-06-07          151676            1.29   
68 2025-06-08             193003 2025-06-08          152211            1.27   

      date_str  
0   01-04-2025  
1   02-04-2025  


In [3]:
##111___Refunds->Refund Related Enquiry->Refund Status Enquiry###

df['session_dt'] = df['session_dt'].astype(str).str.replace(r'[\x00-\x1F\u200B-\u200F\uFEFF\u00A0]', '', regex=True).str.strip()

df["session_dt"] = pd.to_datetime(df["session_dt"], errors="coerce")
df["session_dt_parsed"] = df["session_dt"].dt.date 

merged_df3 = merged_df.copy()
merged_df3["refund_mode"] = merged_df3["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data8 = merged_df3[(~merged_df3["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                            (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                            (merged_df3["JN"]=="REFUNDS") &
                          (merged_df3["ssi"]=="Refunds->Refund Related Enquiry->Refund Status Enquiry")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","FLIPKART_FINANCE","Others"]

filtered_data8["refund_mode"] =pd.Categorical(filtered_data8["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_1_Enquiry = (filtered_data8.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi1_count"}))

refund_Refund_SSS_1_Enquiry["session_dt"] = pd.to_datetime(refund_Refund_SSS_1_Enquiry["session_dt"], errors='coerce')

all_dates1 = filtered_data8["session_dt"].dropna().unique()

all_combinations1 = pd.DataFrame(list(product(transaction_order8, all_dates1)), columns=["refund_mode","session_dt" ])
all_combinations1["session_dt"] = pd.to_datetime(all_combinations1["session_dt"], errors='coerce')

complete_summary1 = all_combinations1.merge(
    refund_Refund_SSS_1_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi1_countact = complete_summary1.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi1_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols1 = sorted(pivot_refund_mode_ssi1_countact.columns)
pivot_refund_mode_ssi1_countact = pivot_refund_mode_ssi1_countact[sorted_cols1]
pivot_refund_mode_ssi1_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols1]
pivot_refund_mode_ssi1_countact.reset_index(inplace=True)

pivot_refund_mode_ssi1_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi1_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi1_countact = pivot_refund_mode_ssi1_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi1_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi1_countact[col] = pivot_refund_mode_ssi1_countact[col] * conversion_map[col]


pivot_refund_mode_ssi1_countact.iloc[:, 1:] = pivot_refund_mode_ssi1_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(pivot_refund_mode_ssi1_countact)


######-----------------------
##222-----Refunds->Refunds->Refund Related Issues->Refund stuck_Int/processing###

merged_df3 = merged_df.copy()
merged_df3["refund_mode"] = merged_df3["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data9 = merged_df3[(~merged_df3["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                            (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                            (merged_df3["JN"]=="REFUNDS") &
                          (merged_df3["ssi"]=="Refunds->Refund Related Issues->Refund stuck_Int/processing")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","Others"]

filtered_data9["refund_mode"] =pd.Categorical(filtered_data9["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_2_Enquiry = (filtered_data9.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi2_count"}))

all_dates2 = filtered_data9["session_dt"].dropna().unique()
all_combinations2 = pd.DataFrame(list(product(transaction_order8, all_dates2)), columns=["refund_mode","session_dt" ])


complete_summary2 = all_combinations1.merge(
    refund_Refund_SSS_2_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi2_countact = complete_summary2.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi2_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols2 = sorted(pivot_refund_mode_ssi2_countact.columns)
pivot_refund_mode_ssi2_countact = pivot_refund_mode_ssi2_countact[sorted_cols2]
pivot_refund_mode_ssi2_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols2]
pivot_refund_mode_ssi2_countact.reset_index(inplace=True)

pivot_refund_mode_ssi2_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi2_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi2_countact = pivot_refund_mode_ssi2_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi2_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi2_countact[col] = pivot_refund_mode_ssi2_countact[col] * conversion_map[col]


pivot_refund_mode_ssi2_countact.iloc[:, 1:] = pivot_refund_mode_ssi2_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(pivot_refund_mode_ssi2_countact)
######-----------------------
##4444---Refunds->Refunds->Refund Related Issues->Refund completed but not reflecting###

merged_df4 = merged_df.copy()
merged_df4["refund_mode"] = merged_df4["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data10 = merged_df3[(~merged_df4["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                              (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                             (merged_df4["JN"]=="REFUNDS") &
                          (merged_df4["ssi"]=="Refunds->Refund Related Issues->Refund completed but not reflecting")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","Others"]

filtered_data10["refund_mode"] =pd.Categorical(filtered_data10["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_4_Enquiry = (filtered_data10.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
.rename(columns={"contacts": "refund_mode_ssi4_count"}))
refund_Refund_SSS_4_Enquiry["session_dt"] = pd.to_datetime(refund_Refund_SSS_4_Enquiry["session_dt"], errors='coerce')

all_dates4 = filtered_data10["session_dt"].dropna().unique()
all_combinations4 = pd.DataFrame(list(product(transaction_order8, all_dates4)), columns=["refund_mode","session_dt" ])
all_combinations4["session_dt"] = pd.to_datetime(all_combinations4["session_dt"], errors='coerce')


complete_summary4 = all_combinations4.merge(
    refund_Refund_SSS_4_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi4_countact = complete_summary4.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi4_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols4 = sorted(pivot_refund_mode_ssi4_countact.columns)
pivot_refund_mode_ssi4_countact = pivot_refund_mode_ssi4_countact[sorted_cols4]
pivot_refund_mode_ssi4_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols4]
pivot_refund_mode_ssi4_countact.reset_index(inplace=True)

pivot_refund_mode_ssi4_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi4_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi4_countact = pivot_refund_mode_ssi4_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi4_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi4_countact[col] = pivot_refund_mode_ssi4_countact[col] * conversion_map[col]


pivot_refund_mode_ssi4_countact.iloc[:, 1:] = pivot_refund_mode_ssi4_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(pivot_refund_mode_ssi4_countact)
######-----------------------
##5555--------Refunds->Refunds->Refund Related Issues->Refund Failed/cancelled###

merged_df5 = merged_df.copy()
merged_df5["refund_mode"] = merged_df5["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data11 = merged_df3[(~merged_df5["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                             (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                             (merged_df5["JN"]=="REFUNDS") &
                          (merged_df5["ssi"]=="Refunds->Refund Related Issues->Refund Failed/cancelled")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","Others"]

filtered_data11["refund_mode"] =pd.Categorical(filtered_data11["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_5_Enquiry = (filtered_data11.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi5_count"}))
refund_Refund_SSS_5_Enquiry["session_dt"] = pd.to_datetime(refund_Refund_SSS_5_Enquiry["session_dt"], errors='coerce')

all_dates5 = filtered_data11["session_dt"].dropna().unique()
all_combinations5 = pd.DataFrame(list(product(transaction_order8, all_dates5)), columns=["refund_mode","session_dt" ])

all_combinations5["session_dt"] = pd.to_datetime(all_combinations5["session_dt"], errors='coerce')

complete_summary5 = all_combinations5.merge(
    refund_Refund_SSS_5_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi5_countact = complete_summary5.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi5_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols5 = sorted(pivot_refund_mode_ssi5_countact.columns)
pivot_refund_mode_ssi5_countact = pivot_refund_mode_ssi5_countact[sorted_cols5]
pivot_refund_mode_ssi5_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols5]
pivot_refund_mode_ssi5_countact.reset_index(inplace=True)

pivot_refund_mode_ssi5_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi5_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi5_countact = pivot_refund_mode_ssi5_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi5_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi5_countact[col] = pivot_refund_mode_ssi5_countact[col] * conversion_map[col]


pivot_refund_mode_ssi5_countact.iloc[:, 1:] = pivot_refund_mode_ssi5_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(pivot_refund_mode_ssi5_countact)
######-----------------------
##666666Refunds->Refund Follow-ups->Gathered Offline Refund Details->NA###

merged_df6 = merged_df.copy()
merged_df6["refund_mode"] = merged_df6["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data12 = merged_df3[(~merged_df6["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                            (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                             (merged_df6["JN"]=="REFUNDS") &
                          (merged_df6["ssi"]=="Refund Follow-ups->Gathered Offline Refund Details->NA")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","Others"]

filtered_data12["refund_mode"] =pd.Categorical(filtered_data12["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_6_Enquiry = (filtered_data12.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi6_count"}))
refund_Refund_SSS_6_Enquiry["session_dt"] = pd.to_datetime(refund_Refund_SSS_6_Enquiry["session_dt"], errors='coerce')

all_dates6 = filtered_data12["session_dt"].dropna().unique()
all_combinations6 = pd.DataFrame(list(product(transaction_order8, all_dates6)), columns=["refund_mode","session_dt" ])
all_combinations6["session_dt"] = pd.to_datetime(all_combinations6["session_dt"], errors='coerce')

complete_summary6 = all_combinations6.merge(
    refund_Refund_SSS_6_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi6_countact = complete_summary6.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi6_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols6 = sorted(pivot_refund_mode_ssi6_countact.columns)
pivot_refund_mode_ssi6_countact = pivot_refund_mode_ssi6_countact[sorted_cols6]
pivot_refund_mode_ssi6_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols6]
pivot_refund_mode_ssi6_countact.reset_index(inplace=True)

pivot_refund_mode_ssi6_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi6_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi6_countact = pivot_refund_mode_ssi6_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi6_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi6_countact[col] = pivot_refund_mode_ssi6_countact[col] * conversion_map[col]


pivot_refund_mode_ssi6_countact.iloc[:, 1:] = pivot_refund_mode_ssi6_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(pivot_refund_mode_ssi6_countact)

######-----------------------
##777Refunds->Refund Related Issues->Cancellation done but refund pending###

merged_df7 = merged_df.copy()
merged_df7["refund_mode"] = merged_df7["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data13 = merged_df7[(~merged_df7["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                             (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                             (merged_df7["JN"]=="REFUNDS") &
                          (merged_df7["ssi"]=="Refunds->Refund Related Issues->Cancellation done but refund pending")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","Others"]

filtered_data13["refund_mode"] =pd.Categorical(filtered_data13["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_7_Enquiry = (filtered_data13.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi7_count"}))
refund_Refund_SSS_7_Enquiry["session_dt"] = pd.to_datetime(refund_Refund_SSS_7_Enquiry["session_dt"], errors='coerce')

all_dates7 = filtered_data13["session_dt"].dropna().unique()
all_combinations7 = pd.DataFrame(list(product(transaction_order8, all_dates7)), columns=["refund_mode","session_dt" ])

all_combinations7["session_dt"] = pd.to_datetime(all_combinations7["session_dt"], errors='coerce')

complete_summary7 = all_combinations7.merge(
    refund_Refund_SSS_7_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi7_countact = complete_summary7.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi7_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols7 = sorted(pivot_refund_mode_ssi7_countact.columns)
pivot_refund_mode_ssi7_countact = pivot_refund_mode_ssi7_countact[sorted_cols7]
pivot_refund_mode_ssi7_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols7]
pivot_refund_mode_ssi7_countact.reset_index(inplace=True)

pivot_refund_mode_ssi7_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi7_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi7_countact = pivot_refund_mode_ssi7_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi7_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi7_countact[col] = pivot_refund_mode_ssi7_countact[col] * conversion_map[col]


pivot_refund_mode_ssi7_countact.iloc[:, 1:] = pivot_refund_mode_ssi7_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(pivot_refund_mode_ssi7_countact)
######-----------------------
##888888Refunds->Flipkart Plus->Plus Price->Issues with Plus Price coins refund###

merged_df8 = merged_df.copy()
merged_df8["refund_mode"] = merged_df8["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data14 = merged_df8[(~merged_df8["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                             (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                             (merged_df8["JN"]=="REFUNDS") &
                          (merged_df8["ssi"]=="Flipkart Plus->Plus Price->Issues with Plus Price coins refund")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","Others"]

filtered_data14["refund_mode"] =pd.Categorical(filtered_data14["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_8_Enquiry = (filtered_data14.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi8_count"}))

refund_Refund_SSS_8_Enquiry["session_dt"] = pd.to_datetime(refund_Refund_SSS_8_Enquiry["session_dt"], errors='coerce')

all_dates8 = filtered_data14["session_dt"].dropna().unique()
all_combinations8 = pd.DataFrame(list(product(transaction_order8, all_dates8)), columns=["refund_mode","session_dt" ])

all_combinations8["session_dt"] = pd.to_datetime(all_combinations8["session_dt"], errors='coerce')

complete_summary8 = all_combinations8.merge(
    refund_Refund_SSS_8_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi8_countact = complete_summary8.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi8_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols8 = sorted(pivot_refund_mode_ssi8_countact.columns)
pivot_refund_mode_ssi8_countact = pivot_refund_mode_ssi8_countact[sorted_cols8]
pivot_refund_mode_ssi8_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols8]
pivot_refund_mode_ssi8_countact.reset_index(inplace=True)

pivot_refund_mode_ssi8_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi8_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi8_countact = pivot_refund_mode_ssi8_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi8_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi8_countact[col] = pivot_refund_mode_ssi8_countact[col] * conversion_map[col]


pivot_refund_mode_ssi_countact.iloc[:, 1:] = pivot_refund_mode_ssi_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(pivot_refund_mode_ssi8_countact)

######-----------------------
##99999Refunds->Refunds->Refund Related Issues->Refund Status Related###

merged_df9 = merged_df.copy()
merged_df9["refund_mode"] = merged_df9["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data15 = merged_df9[(~merged_df9["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                             (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                             (merged_df9["JN"]=="REFUNDS") &
                          (merged_df9["ssi"]=="Refunds->Refund Related Issues->Refund Status Related")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","Others"]

filtered_data15["refund_mode"] =pd.Categorical(filtered_data15["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_9_Enquiry = (filtered_data15.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi9_count"}))

all_dates9 = filtered_data15["session_dt"].dropna().unique()
all_combinations9 = pd.DataFrame(list(product(transaction_order8, all_dates9)), columns=["refund_mode","session_dt" ])


complete_summary9 = all_combinations1.merge(
    refund_Refund_SSS_9_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi9_countact = complete_summary9.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi9_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols9 = sorted(pivot_refund_mode_ssi9_countact.columns)
pivot_refund_mode_ssi9_countact = pivot_refund_mode_ssi9_countact[sorted_cols9]
pivot_refund_mode_ssi9_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols2]
pivot_refund_mode_ssi9_countact.reset_index(inplace=True)

pivot_refund_mode_ssi9_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi9_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi9_countact = pivot_refund_mode_ssi9_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi9_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi9_countact[col] = pivot_refund_mode_ssi9_countact[col] * conversion_map[col]


pivot_refund_mode_ssi9_countact.iloc[:, 1:] = pivot_refund_mode_ssi9_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(pivot_refund_mode_ssi9_countact)

###1000------------------------------------
##!0000Refunds->Refund Related Issues->Return Override###

merged_df10 = merged_df.copy()
merged_df10["refund_mode"] = merged_df10["refund_mode"].apply(
lambda x: "Others" if pd.isna(x) or str(x).strip()== "" else x)

filtered_data16 = merged_df10[(~merged_df10["marketplace_id"].isin(["HYPERLOCAL", "RCBP","FLIPKART"])) &
                                (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                              (merged_df10["JN"]=="REFUNDS") &
                          (merged_df10["ssi"]=="Refunds->Refund Related Issues->Return Override")].copy()   
    
transaction_order8 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
"E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","FLIPKART_FINANCE","Others"]

filtered_data16["refund_mode"] =pd.Categorical(filtered_data16["refund_mode"], categories=transaction_order8, ordered=True)

refund_Refund_SSS_10_Enquiry = (filtered_data16.groupby(["refund_mode","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "refund_mode_ssi10_count"}))

refund_Refund_SSS_10_Enquiry["session_dt"] = pd.to_datetime(refund_Refund_SSS_10_Enquiry["session_dt"], errors='coerce')

all_dates10 = filtered_data16["session_dt"].dropna().unique()
all_combinations10 = pd.DataFrame(list(product(transaction_order8, all_dates10)), columns=["refund_mode","session_dt" ])
all_combinations10["session_dt"] = pd.to_datetime(all_combinations10["session_dt"], errors='coerce')

complete_summary10 = all_combinations10.merge(
    refund_Refund_SSS_10_Enquiry,
    how = "left",
    on=["refund_mode", "session_dt"]
).fillna(0)

pivot_refund_mode_ssi10_countact = complete_summary10.pivot_table(index="refund_mode",
                                                                     columns="session_dt",
                                                                     values = "refund_mode_ssi10_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

sorted_cols10 = sorted(pivot_refund_mode_ssi10_countact.columns)
pivot_refund_mode_ssi10_countact = pivot_refund_mode_ssi10_countact[sorted_cols10]
pivot_refund_mode_ssi10_countact.columns = [col.strftime('%d-%m-%Y') for col in sorted_cols10]
pivot_refund_mode_ssi10_countact.reset_index(inplace=True)

pivot_refund_mode_ssi10_countact["refund_mode"]=pd.Categorical(
	pivot_refund_mode_ssi10_countact["refund_mode"],
	categories = transaction_order8,
	ordered =True)

pivot_refund_mode_ssi10_countact = pivot_refund_mode_ssi10_countact.sort_values("refund_mode").reset_index(drop=True)	

for col in pivot_refund_mode_ssi10_countact.columns[1:]:
    if col in conversion_map:
        pivot_refund_mode_ssi10_countact[col] = pivot_refund_mode_ssi10_countact[col] * conversion_map[col]


pivot_refund_mode_ssi10_countact.iloc[:, 1:] = pivot_refund_mode_ssi10_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

##print(pivot_refund_mode_ssi10_countact)


output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Grocery_Refund_SSI_.xlsx"
with pd.ExcelWriter(output_path, engine = 'openpyxl') as writer:
    pivot_refund_mode_ssi1_countact.to_excel(writer, sheet_name ='Status_Enquiry', index=False)
    pivot_refund_mode_ssi2_countact.to_excel(writer, sheet_name ='processing', index=False )
    pivot_refund_mode_ssi4_countact.to_excel(writer, sheet_name ='not reflecting', index=False )
    pivot_refund_mode_ssi5_countact.to_excel(writer, sheet_name = 'cancelled', index=False)
    pivot_refund_mode_ssi6_countact.to_excel(writer, sheet_name = 'Details->NA', index=False)
    pivot_refund_mode_ssi7_countact.to_excel(writer, sheet_name = 'pending', index=False)
    pivot_refund_mode_ssi8_countact.to_excel(writer, sheet_name = 'coins refund', index=False)
    pivot_refund_mode_ssi9_countact.to_excel(writer, sheet_name = 'Status Related', index=False)
    pivot_refund_mode_ssi10_countact.to_excel(writer, sheet_name = 'Return Override', index=False)
print("Excel file successfully saved:",output_path)


Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Grocery_Refund_SSI_.xlsx


In [62]:
print(merged_df["session_dt"].dtype)
print(filtered_data9["session_dt"].dtype)
print(refund_Refund_SSS_2_Enquiry["session_dt"].dtype)
print(all_combinations2["session_dt"].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]
object
